In [1]:
!pip install -U llama-parse
!pip install jedi jiwer rouge-score sentence-transformers llama-index
!pip install torch==2.1.0
!pip install --upgrade torch torchvision torchaudio
!pip install --upgrade llama-index

  Using cached jedi-0.19.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=2c5abf54d0a970893887befcde5fda58b64281d2b6426f572485e48a38e3fbbb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.1 M

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Saving 68 sidor.pdf to 68 sidor.pdf


In [11]:
import os, nest_asyncio, asyncio
from llama_parse import LlamaParse
from pathlib import Path

nest_asyncio.apply()

# API-nyckel
os.environ["LLAMA_CLOUD_API_KEY"] = "llamaparseAPI"

In [ ]:
# Initiera parsern
parser = LlamaParse(api_key=os.environ["LLAMA_CLOUD_API_KEY"])

llama_results = {}

# Kör LlamaParse på PDF
async def run_llama():
    file_path = "/content/68 sidor.pdf"
    result = await parser.aparse(file_path)

    # Kombinera alla sidor till en sträng
    llama_results[file_path] = "\n".join([page.text for page in result.pages])

await run_llama()

# Hämta extraherad text
markdown_text = llama_results["/content/68 sidor.pdf"]

if not isinstance(markdown_text, str) or not markdown_text.strip():
    raise ValueError("Kunde inte extrahera Markdown från LlamaParse-resultatet.")

# Spara originalet
Path("llamaparse_output.md").write_text(markdown_text, encoding="utf-8")

# Lägg till PAGEBREAKS
md = Path("llamaparse_output.md").read_text()

pages = md.split("\f")  # form feed

# Om inga \f hittas → fallback
if len(pages) == 1:
    pages = md.split("\n\n\n")

new_md = []
for i, page in enumerate(pages, start=1):
    new_md.append(f"# Page {i}\n{page.strip()}\n")

Path("llamaparse_output_paged.md").write_text("\n".join(new_md), encoding="utf-8")

# Ladda ner
files.download("llamaparse_output_paged.md")


In [ ]:
# Initiera parsern
parser = LlamaParse(api_key=os.environ["LLAMA_CLOUD_API_KEY"])

llama_results = {}

# Kör LlamaParse på PDF
async def run_llama():
    file_path = "/content/68 sidor.pdf"
    result = await parser.aparse(file_path)

    # ✅ Spara sidorna som en lista istället för att joina dem
    llama_results[file_path] = result.pages  # <-- lista av page-objekt

await run_llama()

pages = llama_results["/content/68 sidor.pdf"]

if not pages:
    raise ValueError("Kunde inte extrahera sidor från LlamaParse-resultatet.")

# Bygg markdown med en section per sida
new_md = []
for i, page in enumerate(pages, start=1):
    text = page.text.strip()
    new_md.append(f"# Page {i}\n\n{text}\n\n---\n")

markdown_output = "\n".join(new_md)

# Spara
Path("llamaparse_claude_output_paged.md").write_text(markdown_output, encoding="utf-8")

# Ladda ner
files.download("llamaparse_claude_output_paged.md")

Started parsing the file under job_id 01285041-ee19-42eb-80b8-2a49c19e2d75
...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>